In [1]:
import torch
print(torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset

MODEL = "/kaggle/input/models/aisuko/xlm-roberta-base/transformers/default/1"
DATA   = "/kaggle/input/competitions/rmit-hackathon-2026"

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I    = {l: i for i, l in enumerate(LABELS)}

MAXLEN, EPOCHS, LR = 256, 1, 2e-5
BS_PER_GPU = 16          # 2 x T4 -> effective train batch 32
NGPU = max(torch.cuda.device_count(), 1)

train = pd.read_csv(f"{DATA}/train.csv")[:200]
test  = pd.read_csv(f"{DATA}/test.csv")

tr, va = train_test_split(train, test_size=0.1, stratify=train.label, random_state=42)

tok = AutoTokenizer.from_pretrained(MODEL)

def mk(df, labelled=True):
    d = df[["text"]].copy()
    if labelled:
        d["labels"] = df.label.map(L2I).values
    return (Dataset.from_pandas(d, preserve_index=False)
            .map(lambda b: tok(b["text"], truncation=True, max_length=MAXLEN),
                 batched=True, remove_columns=["text"]))

ds_tr, ds_va, ds_te = mk(tr), mk(va), mk(test, labelled=False)
print(len(ds_tr), len(ds_va), len(ds_te))

2 ['Tesla T4', 'Tesla T4']


Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

180 20 6000


In [2]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, 
    num_labels=len(LABELS),
    id2label={i: l for l, i in L2I.items()}, 
    label2id=L2I
)

print(model)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: /kaggle/input/models/aisuko/xlm-roberta-base/transformers/default/1
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=3, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li